# Incremental value of PRS and NMR score


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index

try:
    from sksurv.metrics import concordance_index_ipcw
    from sksurv.util import Surv
except ImportError as error:
    raise ImportError(
        "scikit-survival is required for Uno IPCW C-index and Coxnet. "
        "Install it in the server environment with: uv pip install scikit-survival"
    ) from error

try:
    from dcurves import dca as dcurves_dca
except ImportError as error:
    raise ImportError(
        "dcurves is required for standard survival DCA. "
        "Install it in the server environment with: uv pip install 'dcurves>=1.1.7'"
    ) from error

CNS = {
    "blue": "#3C5488",
    "red": "#E64B35",
    "green": "#00A087",
    "teal": "#4DBBD5",
    "orange": "#F39B7F",
    "purple": "#8491B4",
    "gray": "#6E6E6E",
    "light_gray": "#D9D9D9",
    "black": "#222222",
}

plt.rcParams.update({
    "figure.dpi": 170,
    "savefig.dpi": 600,
    "font.family": "DejaVu Sans",
    "font.size": 13.0,
    "axes.titlesize": 15.0,
    "axes.labelsize": 13.5,
    "xtick.labelsize": 12.0,
    "ytick.labelsize": 12.0,
    "legend.fontsize": 11.0,
    "axes.linewidth": 0.9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

HORIZONS = [5.0, 10.0]
NRI_CUTPOINTS = [
    float(value)
    for value in os.environ.get("CHD_NRI_CUTPOINTS", "0.05,0.10,0.20").split(",")
]
NRI_CUTPOINT_SOURCE = os.environ.get("CHD_NRI_CUTPOINT_SOURCE", "").strip()
if sorted(NRI_CUTPOINTS) != NRI_CUTPOINTS or any(
    value <= 0 or value >= 1 for value in NRI_CUTPOINTS
):
    raise ValueError("CHD_NRI_CUTPOINTS must be strictly increasing probabilities in (0, 1).")
CATEGORICAL_NRI_ENABLED = bool(NRI_CUTPOINT_SOURCE)
if not CATEGORICAL_NRI_ENABLED:
    warnings.warn("CHD_NRI_CUTPOINT_SOURCE is unset: categorical NRI will be omitted; continuous NRI and IDI remain active.")
BOOTSTRAP_N = int(os.environ.get("CHD_BOOTSTRAP_N", "1000"))
BOOTSTRAP_SEED = int(os.environ.get("CHD_BOOTSTRAP_SEED", "2026"))
IBS_GRID_N = int(os.environ.get("CHD_IBS_GRID_N", "51"))
DCA_THRESHOLDS = np.arange(0.01, 0.201, 0.005)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return str(path)

def clean_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

def save_figure(fig, output_dir, stem):
    ensure_dir(output_dir)
    for extension in ["png", "pdf", "svg"]:
        fig.savefig(
            os.path.join(output_dir, f"{stem}.{extension}"),
            dpi=600,
            bbox_inches="tight",
        )
    print("Saved figure:", os.path.join(output_dir, stem + ".png"))

def quantile_ci(values, lower=0.025, upper=0.975):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    return float(np.quantile(values, lower)), float(np.quantile(values, upper))

def make_surv(frame):
    return Surv.from_arrays(
        event=frame["Event"].astype(bool).to_numpy(),
        time=frame["Time"].astype(float).to_numpy(),
    )

def eid_digest(frame):
    if "Eid" not in frame.columns:
        return "no_eid"
    values = pd.util.hash_pandas_object(frame["Eid"], index=False).to_numpy()
    return hashlib.sha256(values.tobytes()).hexdigest()

def harrell_c(frame, risk):
    return float(
        concordance_index(
            frame["Time"].astype(float).to_numpy(),
            -np.asarray(risk, dtype=float).reshape(-1),
            frame["Event"].astype(int).to_numpy(),
        )
    )

def uno_c(train_frame, eval_frame, risk, tau):
    if float(tau) <= 0:
        raise ValueError("Uno C tau must be positive.")
    estimate = concordance_index_ipcw(
        make_surv(train_frame),
        make_surv(eval_frame),
        np.asarray(risk, dtype=float).reshape(-1),
        tau=float(tau),
    )[0]
    return float(estimate)

def paired_bootstrap_uno(
    train_frame,
    eval_frame,
    risk_new,
    risk_reference,
    tau,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    risk_new = np.asarray(risk_new, dtype=float).reshape(-1)
    risk_reference = np.asarray(risk_reference, dtype=float).reshape(-1)
    original_new = uno_c(train_frame, eval_frame, risk_new, tau)
    original_reference = uno_c(train_frame, eval_frame, risk_reference, tau)
    original_delta = original_new - original_reference

    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        if boot_frame["Event"].nunique() < 2:
            continue
        try:
            values.append(
                uno_c(train_frame, boot_frame, risk_new[idx], tau)
                - uno_c(train_frame, boot_frame, risk_reference[idx], tau)
            )
        except Exception:
            continue
    lower, upper = quantile_ci(values)
    return {
        "Uno_C_New": original_new,
        "Uno_C_Reference": original_reference,
        "Delta_Uno_C": original_delta,
        "Delta_Uno_C_Lower95": lower,
        "Delta_Uno_C_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def paired_bootstrap_harrell(
    eval_frame,
    risk_new,
    risk_reference,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    risk_new = np.asarray(risk_new, dtype=float).reshape(-1)
    risk_reference = np.asarray(risk_reference, dtype=float).reshape(-1)
    original_new = harrell_c(eval_frame, risk_new)
    original_reference = harrell_c(eval_frame, risk_reference)
    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        if boot_frame["Event"].nunique() < 2:
            continue
        try:
            values.append(
                harrell_c(boot_frame, risk_new[idx])
                - harrell_c(boot_frame, risk_reference[idx])
            )
        except Exception:
            continue
    lower, upper = quantile_ci(values)
    return {
        "Harrell_C_New": original_new,
        "Harrell_C_Reference": original_reference,
        "Delta_Harrell_C": original_new - original_reference,
        "Delta_Harrell_C_Lower95": lower,
        "Delta_Harrell_C_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def fit_absolute_risk_calibrator(train_frame, train_risk, label):
    train_risk=np.asarray(train_risk,dtype=float).reshape(-1)
    center=float(np.mean(train_risk)); scale=float(np.std(train_risk,ddof=0))
    if not np.isfinite(scale) or scale<=0: raise ValueError(f"Degenerate training risk for {label}")
    calibration_frame=pd.DataFrame({"Time":train_frame["Time"].astype(float).to_numpy(),"Event":train_frame["Event"].astype(int).to_numpy(),"Risk_Score":(train_risk-center)/scale})
    calibrator=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        calibrator.fit(calibration_frame,duration_col="Time",event_col="Event",show_progress=False)
    return {"label":label,"model":calibrator,"center":center,"scale":scale,"method":"train-fitted Cox recalibration of standardized raw model score"}

def predict_absolute_risk(calibrator, raw_risk, horizon):
    score=(np.asarray(raw_risk,dtype=float).reshape(-1)-calibrator["center"])/calibrator["scale"]
    survival=calibrator["model"].predict_survival_function(pd.DataFrame({"Risk_Score":score}),times=[float(horizon)])
    return np.clip(1.0-survival.iloc[0].to_numpy(dtype=float),0.0,1.0)

def prediction_matrix(calibrator, raw_risk, times):
    return np.column_stack([
        predict_absolute_risk(calibrator, raw_risk, float(time))
        for time in times
    ])

def fit_censoring_km(frame):
    km = KaplanMeierFitter()
    km.fit(
        frame["Time"].astype(float).to_numpy(),
        event_observed=1 - frame["Event"].astype(int).to_numpy(),
        label="G_censoring",
    )
    return km

def censor_survival(km, times, left_limit=False, minimum=1e-8):
    times = np.asarray(times, dtype=float)
    if left_limit:
        times = np.nextafter(times, -np.inf)
        times = np.maximum(times, 0.0)
    values = np.asarray(km.predict(times), dtype=float).reshape(-1)
    return np.clip(values, minimum, 1.0)

def horizon_support_audit(train_frame, eval_frame, horizons=HORIZONS):
    train_censoring_km = fit_censoring_km(train_frame)
    eval_censoring_km = fit_censoring_km(eval_frame)
    rows = []
    for horizon in horizons:
        horizon = float(horizon)
        train_time = train_frame["Time"].astype(float).to_numpy()
        train_event = train_frame["Event"].astype(int).to_numpy()
        eval_time = eval_frame["Time"].astype(float).to_numpy()
        eval_event = eval_frame["Event"].astype(int).to_numpy()
        rows.append({
            "Horizon_Years": horizon,
            "Train_Event_By_Horizon_N": int(((train_event == 1) & (train_time <= horizon)).sum()),
            "Train_Observed_Beyond_Horizon_N": int((train_time > horizon).sum()),
            "Test_Event_By_Horizon_N": int(((eval_event == 1) & (eval_time <= horizon)).sum()),
            "Test_Observed_Beyond_Horizon_N": int((eval_time > horizon).sum()),
            "Train_Censoring_Survival_Gt": float(censor_survival(train_censoring_km, [horizon])[0]),
            "Test_Censoring_Survival_Gt": float(censor_survival(eval_censoring_km, [horizon])[0]),
            "Train_Max_Followup": float(np.nanmax(train_time)),
            "Test_Max_Followup": float(np.nanmax(eval_time)),
        })
    audit = pd.DataFrame(rows)
    unsupported = audit.loc[
        (audit["Train_Event_By_Horizon_N"] == 0)
        | (audit["Train_Observed_Beyond_Horizon_N"] == 0)
        | (audit["Test_Event_By_Horizon_N"] == 0)
        | (audit["Test_Observed_Beyond_Horizon_N"] == 0)
        | (audit["Train_Censoring_Survival_Gt"] <= 1e-4)
        | (audit["Test_Censoring_Survival_Gt"] <= 1e-4)
    ]
    if not unsupported.empty:
        raise RuntimeError(
            "Insufficient event/censoring support at one or more requested horizons:\n"
            + unsupported.to_string(index=False)
        )
    return audit

def ipcw_nri_idi_once(
    eval_frame,
    pred_reference,
    pred_new,
    horizon,
    cutpoints=NRI_CUTPOINTS,
    censoring_km=None,
):
    pred_reference = np.asarray(pred_reference, dtype=float).reshape(-1)
    pred_new = np.asarray(pred_new, dtype=float).reshape(-1)
    time = eval_frame["Time"].astype(float).to_numpy()
    event = eval_frame["Event"].astype(int).to_numpy()
    horizon = float(horizon)
    km = fit_censoring_km(eval_frame) if censoring_km is None else censoring_km

    cases = (event == 1) & (time <= horizon)
    controls = time > horizon
    if cases.sum() == 0 or controls.sum() == 0:
        raise RuntimeError(f"No IPCW cases or controls at horizon={horizon}.")

    case_weights = np.zeros(len(eval_frame), dtype=float)
    control_weights = np.zeros(len(eval_frame), dtype=float)
    case_weights[cases] = 1.0 / censor_survival(km, time[cases], left_limit=True)
    g_horizon = float(censor_survival(km, [horizon])[0])
    control_weights[controls] = 1.0 / g_horizon

    case_weights /= case_weights.sum()
    control_weights /= control_weights.sum()
    delta_prediction = pred_new - pred_reference

    case_up = float(np.sum(case_weights * (delta_prediction > 0)))
    case_down = float(np.sum(case_weights * (delta_prediction < 0)))
    control_up = float(np.sum(control_weights * (delta_prediction > 0)))
    control_down = float(np.sum(control_weights * (delta_prediction < 0)))
    continuous_nri = (case_up - case_down) + (control_down - control_up)

    bins = np.asarray([-np.inf] + list(cutpoints) + [np.inf], dtype=float)
    category_reference = np.digitize(pred_reference, bins, right=False)
    category_new = np.digitize(pred_new, bins, right=False)
    category_delta = category_new - category_reference
    categorical_nri = (
        float(np.sum(case_weights * (category_delta > 0)))
        - float(np.sum(case_weights * (category_delta < 0)))
        + float(np.sum(control_weights * (category_delta < 0)))
        - float(np.sum(control_weights * (category_delta > 0)))
    )

    if not NRI_CUTPOINT_SOURCE:
        categorical_nri = np.nan

    discrimination_new = (
        float(np.sum(case_weights * pred_new))
        - float(np.sum(control_weights * pred_new))
    )
    discrimination_reference = (
        float(np.sum(case_weights * pred_reference))
        - float(np.sum(control_weights * pred_reference))
    )
    idi = discrimination_new - discrimination_reference
    return {
        "NRI_Categorical": categorical_nri,
        "NRI_Continuous": continuous_nri,
        "IDI": idi,
        "Censoring_Survival_At_Horizon": g_horizon,
        "Case_N": int(cases.sum()),
        "Control_N": int(controls.sum()),
        "Censored_Before_Horizon_N": int((~cases & ~controls).sum()),
    }

def bootstrap_ipcw_nri_idi(
    eval_frame,
    pred_reference,
    pred_new,
    horizon,
    cutpoints=NRI_CUTPOINTS,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    original = ipcw_nri_idi_once(
        eval_frame,
        pred_reference,
        pred_new,
        horizon,
        cutpoints=cutpoints,
    )
    pred_reference = np.asarray(pred_reference, dtype=float).reshape(-1)
    pred_new = np.asarray(pred_new, dtype=float).reshape(-1)
    rng = np.random.default_rng(seed)
    n = len(eval_frame)
    values = {"NRI_Categorical": [], "NRI_Continuous": [], "IDI": []}
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        boot_frame = eval_frame.iloc[idx].reset_index(drop=True)
        try:
            estimate = ipcw_nri_idi_once(
                boot_frame,
                pred_reference[idx],
                pred_new[idx],
                horizon,
                cutpoints=cutpoints,
            )
            for metric in values:
                values[metric].append(estimate[metric])
        except Exception:
            continue

    output = dict(original)
    for metric, metric_values in values.items():
        lower, upper = quantile_ci(metric_values)
        output[f"{metric}_Lower95"] = lower
        output[f"{metric}_Upper95"] = upper
    output["Bootstrap_N_Used"] = int(min(len(v) for v in values.values()))
    return output

def subject_level_ipcw_ibs(
    train_frame,
    eval_frame,
    calibrator,
    eval_risk,
    horizon,
    grid_n=IBS_GRID_N,
):
    horizon = float(horizon)
    times = np.linspace(0.0, horizon, int(grid_n))
    predicted_risk = prediction_matrix(calibrator, eval_risk, times)
    time = eval_frame["Time"].astype(float).to_numpy()
    event = eval_frame["Event"].astype(int).to_numpy()
    censor_km = fit_censoring_km(train_frame)
    contributions = np.zeros_like(predicted_risk, dtype=float)

    for j, point in enumerate(times):
        y = ((event == 1) & (time <= point)).astype(float)
        weights = np.zeros(len(eval_frame), dtype=float)
        cases = (event == 1) & (time <= point)
        controls = time > point
        if cases.any():
            weights[cases] = 1.0 / censor_survival(
                censor_km,
                time[cases],
                left_limit=True,
            )
        if controls.any():
            weights[controls] = 1.0 / float(censor_survival(censor_km, [point])[0])
        contributions[:, j] = weights * (y - predicted_risk[:, j]) ** 2

    if hasattr(np, "trapezoid"):
        subject_ibs = np.trapezoid(contributions, times, axis=1) / horizon
    else:
        subject_ibs = np.trapz(contributions, times, axis=1) / horizon
    return {
        "IBS": float(np.mean(subject_ibs)),
        "Subject_IBS": subject_ibs,
        "Times": times,
    }

def paired_bootstrap_delta_ibs(
    new_ibs,
    reference_ibs,
    bootstrap_n=BOOTSTRAP_N,
    seed=BOOTSTRAP_SEED,
):
    paired = (
        np.asarray(new_ibs["Subject_IBS"], dtype=float)
        - np.asarray(reference_ibs["Subject_IBS"], dtype=float)
    )
    rng = np.random.default_rng(seed)
    n = len(paired)
    values = []
    for _ in range(int(bootstrap_n)):
        idx = rng.integers(0, n, size=n)
        values.append(float(np.mean(paired[idx])))
    lower, upper = quantile_ci(values)
    return {
        "IBS_New": float(new_ibs["IBS"]),
        "IBS_Reference": float(reference_ibs["IBS"]),
        "Delta_IBS": float(np.mean(paired)),
        "Delta_IBS_Lower95": lower,
        "Delta_IBS_Upper95": upper,
        "Bootstrap_N_Used": int(len(values)),
    }

def km_observed_risk(frame, horizon):
    if len(frame) == 0:
        return np.nan
    km = KaplanMeierFitter()
    km.fit(
        frame["Time"].astype(float).to_numpy(),
        event_observed=frame["Event"].astype(int).to_numpy(),
    )
    return float(1.0 - km.predict(float(horizon)))

def calibration_by_quantile(eval_frame, predicted_risk, horizon, groups=10):
    temporary = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    temporary["Predicted_Risk"] = np.asarray(predicted_risk, dtype=float).reshape(-1)
    temporary["Risk_Group"] = pd.qcut(
        temporary["Predicted_Risk"].rank(method="first"),
        q=int(groups),
        labels=False,
        duplicates="drop",
    )
    rows = []
    for group, data in temporary.groupby("Risk_Group"):
        rows.append({
            "Risk_Group": int(group) + 1,
            "N": int(len(data)),
            "Predicted_Risk": float(data["Predicted_Risk"].mean()),
            "Observed_KM_Risk": km_observed_risk(data, horizon),
        })
    return pd.DataFrame(rows)

def _safe_dca_column(name):
    text = re.sub(r"[^A-Za-z0-9_]+", "_", str(name)).strip("_")
    return "model_" + text.lower()

def survival_dca(eval_frame, prediction_dict, horizon, thresholds=DCA_THRESHOLDS):
    data = eval_frame[["Time", "Event"]].reset_index(drop=True).copy()
    model_columns = []
    label_map = {}
    for label, prediction in prediction_dict.items():
        column = _safe_dca_column(label)
        data[column] = np.asarray(prediction, dtype=float).reshape(-1)
        model_columns.append(column)
        label_map[column] = label

    result = dcurves_dca(
        data=data,
        outcome="Event",
        modelnames=model_columns,
        thresholds=np.asarray(thresholds, dtype=float),
        time_to_outcome_col="Time",
        time=float(horizon),
    ).copy()
    result.columns = [str(column).lower() for column in result.columns]
    rename = {}
    for candidate in ["model", "models", "label"]:
        if candidate in result.columns:
            rename[candidate] = "Strategy"
            break
    for candidate in ["threshold", "thresholds"]:
        if candidate in result.columns:
            rename[candidate] = "Threshold"
            break
    for candidate in ["net_benefit", "net benefit", "netbenefit"]:
        if candidate in result.columns:
            rename[candidate] = "Net_Benefit"
            break
    result = result.rename(columns=rename)
    required = {"Strategy", "Threshold", "Net_Benefit"}
    if not required.issubset(result.columns):
        raise RuntimeError(
            "Unexpected dcurves output columns: " + ", ".join(result.columns)
        )
    result["Strategy"] = result["Strategy"].replace(label_map).replace({
        "all": "Treat all",
        "none": "Treat none",
        "All": "Treat all",
        "None": "Treat none",
    })
    result["Horizon_Years"] = float(horizon)
    result["Split"] = "test"
    result["DCA_Method"] = "dcurves survival DCA; no external IPCW multiplication"
    return result

def draw_forest(
    frame,
    estimate,
    lower,
    upper,
    label,
    title,
    xlabel,
    output_dir,
    stem,
    color="#3C5488",
    null=0.0,
    figsize=(8.2, 4.6),
):
    plot = frame.reset_index(drop=True).copy()
    fig, ax = plt.subplots(figsize=figsize)
    y = np.arange(len(plot))
    ax.axvline(float(null), color=CNS["gray"], lw=1.0, zorder=0)
    for i, row in plot.iterrows():
        value = float(row[estimate])
        low = float(row[lower])
        high = float(row[upper])
        row_color = row[color] if isinstance(color, str) and color in plot.columns else color
        ax.errorbar(
            value,
            i,
            xerr=[[max(0.0, value - low)], [max(0.0, high - value)]],
            fmt="o",
            color=row_color,
            ecolor=row_color,
            elinewidth=1.5,
            capsize=3,
            markersize=6.5,
        )
    ax.set_yticks(y)
    ax.set_yticklabels(plot[label].astype(str))
    ax.invert_yaxis()
    ax.set_xlabel(xlabel)
    ax.set_title(title, loc="left", fontweight="bold")
    ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
    clean_axes(ax)
    fig.tight_layout()
    save_figure(fig, output_dir, stem)
    plt.show()

def draw_dca(dca_frame, title, output_dir, stem, colors, figsize=(8.2, 5.0)):
    fig, ax = plt.subplots(figsize=figsize)
    preferred_order = list(colors)
    available = list(dict.fromkeys(dca_frame["Strategy"].astype(str)))
    order = [item for item in preferred_order if item in available]
    order += [item for item in available if item not in order]
    for strategy in order:
        data = dca_frame.loc[dca_frame["Strategy"].astype(str) == strategy].sort_values("Threshold")
        if data.empty:
            continue
        style = "--" if strategy == "Treat all" else ":" if strategy == "Treat none" else "-"
        width = 1.5 if strategy in {"Treat all", "Treat none"} else 2.1
        ax.plot(
            data["Threshold"],
            data["Net_Benefit"],
            color=colors.get(strategy, CNS["gray"]),
            linestyle=style,
            linewidth=width,
            label=strategy,
        )
    ax.set_xlabel("Predicted-risk threshold")
    ax.set_ylabel("Net benefit")
    ax.set_title(title, loc="left", fontweight="bold")
    ax.grid(color=CNS["light_gray"], lw=0.8)
    ax.legend(frameon=False, fontsize=11.0, ncol=2)
    clean_axes(ax)
    fig.tight_layout()
    save_figure(fig, output_dir, stem)
    plt.show()


In [ ]:
from pathlib import Path
import os

SHARED_NOTEBOOK = os.environ.get(
    "CHD_SHARED_NOTEBOOK",
    str(Path.cwd() / "00_shared_config.ipynb"),
)
if not Path(SHARED_NOTEBOOK).exists():
    raise FileNotFoundError(
        f"Shared notebook not found: {SHARED_NOTEBOOK}. "
        "Place this notebook beside 00_shared_config.ipynb "
        "or set CHD_SHARED_NOTEBOOK."
    )
get_ipython().run_line_magic("run", f'"{SHARED_NOTEBOOK}"')

(
    df_train,
    df_val,
    df_test,
    final_feature_sets,
    pipeline_manifest,
) = validate_pipeline_artifacts(load_frames=True)

RESULT2_ROOT = ensure_dir(os.path.join(
    SENSITIVITY_OUTPUT_DIR,
    "04_results",
    "result2_ipcw_standalone",
))
RESULT2_FIGURE_DIR = ensure_dir(os.path.join(RESULT2_ROOT, "figures"))
RESULT2_SOURCE_DIR = ensure_dir(os.path.join(RESULT2_ROOT, "source_data"))
print("Result 2 output:", RESULT2_ROOT)


In [ ]:
# Fit the four prespecified unpenalized Cox PH models.
# Execution-only optimization: independent feature sets run in parallel and
# lifelines batch mode is enabled. The statistical specification is unchanged:
# penalizer=0, l1_ratio=0, Efron ties, and robust standard errors.

from joblib import Parallel, delayed
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
import warnings

RESULT2_COX_FEATURES = [
    "Clinical_only",
    "Clinical_plus_PRS",
    "Clinical_plus_NMR_score",
    "Clinical_plus_PRS_plus_NMR_score",
]

RESULT2_COX_N_JOBS = min(
    max(1, int(os.environ.get("CHD_RESULT2_COX_N_JOBS", "4"))),
    len(RESULT2_COX_FEATURES),
)

def _fit_one_result2_cox(feature_set_name):
    X_train, X_val, X_test, actual_cols = prepare_X_train_val_test(
        df_train,
        df_val,
        df_test,
        final_feature_sets[feature_set_name],
        scale=True,
    )

    # These values intentionally match the shared signed-cache contract.
    hyperparameters = {
        "penalizer": 0.0,
        "l1_ratio": 0.0,
        "robust_se": True,
        "ties": "efron",
    }
    code_tag = "unpenalized_cox_v2"

    if risk_cache_matches(
        "Cox PH",
        feature_set_name,
        actual_cols,
        hyperparameters,
        code_tag,
    ):
        log_progress(
            f"RESUME Cox PH | {feature_set_name} | signed cache valid"
        )
        return (
            pd.read_csv(model_result_path("Cox PH", feature_set_name))
            .iloc[0]
            .to_dict()
        )

    fit_frame = pd.concat(
        [
            df_train[["Time", "Event"]].reset_index(drop=True),
            X_train.reset_index(drop=True),
        ],
        axis=1,
    )

    log_progress(
        f"START Cox PH | {feature_set_name} | "
        f"features={len(actual_cols)} | parallel_workers={RESULT2_COX_N_JOBS}"
    )

    model = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
    with warnings.catch_warnings():
        warnings.simplefilter("error", ConvergenceWarning)
        model.fit(
            fit_frame,
            duration_col="Time",
            event_col="Event",
            robust=True,
            batch_mode=True,
            show_progress=False,
        )

    risks = {
        "train_risk": model.predict_log_partial_hazard(X_train).to_numpy().reshape(-1),
        "val_risk": model.predict_log_partial_hazard(X_val).to_numpy().reshape(-1),
        "test_risk": model.predict_log_partial_hazard(X_test).to_numpy().reshape(-1),
    }

    signature = save_risk_record(
        "Cox PH",
        feature_set_name,
        risks,
        actual_cols,
        hyperparameters,
        code_tag,
    )

    row = make_result_row(
        MAIN_SCENARIO_NAME,
        "Cox PH",
        feature_set_name,
        actual_cols,
        risks["train_risk"],
        risks["val_risk"],
        risks["test_risk"],
    )
    row.update({
        "COX_PENALIZER": 0.0,
        "Robust_SE": True,
        "Model_Signature": signature,
    })

    atomic_write_csv(
        pd.DataFrame([row]),
        model_result_path("Cox PH", feature_set_name),
        index=False,
    )

    summary = model.summary.reset_index()
    summary = summary.rename(columns={summary.columns[0]: "Variable"})
    summary.insert(0, "Feature_Set", feature_set_name)
    atomic_write_csv(
        summary,
        os.path.join(
            model_result_dir("Cox PH"),
            f"{safe_filename(feature_set_name)}_coefficients.csv",
        ),
        index=False,
    )

    atomic_write_csv(
        model.baseline_cumulative_hazard_.reset_index(),
        os.path.join(
            model_result_dir("Cox PH"),
            f"{safe_filename(feature_set_name)}_baseline_cumulative_hazard.csv",
        ),
        index=False,
    )

    log_progress(
        f"DONE Cox PH | {feature_set_name} | "
        f"test Harrell C={row['Test_C_index']:.6f}"
    )
    return row

log_progress(
    f"Result2 parallel Cox fitting | models={len(RESULT2_COX_FEATURES)} | "
    f"workers={RESULT2_COX_N_JOBS}"
)

_result2_rows = Parallel(
    n_jobs=RESULT2_COX_N_JOBS,
    backend="loky",
    verbose=10,
)(
    delayed(_fit_one_result2_cox)(feature_set_name)
    for feature_set_name in RESULT2_COX_FEATURES
)

result2_cox_fit = pd.DataFrame(_result2_rows)
result2_cox_fit["_Feature_Order"] = pd.Categorical(
    result2_cox_fit["Feature_Set"],
    categories=RESULT2_COX_FEATURES,
    ordered=True,
)
result2_cox_fit = (
    result2_cox_fit
    .sort_values("_Feature_Order")
    .drop(columns="_Feature_Order")
    .reset_index(drop=True)
)

atomic_write_csv(
    result2_cox_fit,
    os.path.join(TABLE_DIR, "result2_cox_model_performance.csv"),
    index=False,
)

_joint_path = os.path.join(
    model_result_dir("Cox PH"),
    "Clinical_plus_PRS_plus_NMR_score_coefficients.csv",
)
_joint = pd.read_csv(_joint_path)
_joint = _joint.loc[
    _joint["Variable"].astype(str).isin([PRS_COL, "NMR_score"])
].copy()
_joint = _joint.rename(columns={
    "exp(coef)": "HR",
    "exp(coef) lower 95%": "HR_CI_Lower",
    "exp(coef) upper 95%": "HR_CI_Upper",
})
if "Feature_Set" not in _joint.columns:
    _joint.insert(0, "Feature_Set", "Clinical_plus_PRS_plus_NMR_score")
if len(_joint) != 2:
    raise RuntimeError(
        "Joint Cox model must yield exactly PRS and NMR_score rows"
    )

atomic_write_csv(
    _joint[[
        "Feature_Set",
        "Variable",
        "HR",
        "HR_CI_Lower",
        "HR_CI_Upper",
        "p",
    ]],
    os.path.join(TABLE_DIR, "result2_cox_prs_nmrscore_hr.csv"),
    index=False,
)

display(result2_cox_fit)
display(_joint)


In [ ]:
RESULT2_FEATURES = [
    "Clinical_only",
    "Clinical_plus_PRS",
    "Clinical_plus_NMR_score",
    "Clinical_plus_PRS_plus_NMR_score",
]
missing_feature_sets = [name for name in RESULT2_FEATURES if name not in final_feature_sets]
if missing_feature_sets:
    raise KeyError("Missing Stage 01 feature sets: " + ", ".join(missing_feature_sets))

selected_feature_sets = {name: final_feature_sets[name] for name in RESULT2_FEATURES}
risk_store = load_all_risk_records(selected_feature_sets, ["Cox PH"])

def result2_record(feature_set):
    key = (MAIN_SCENARIO_NAME, feature_set, "Cox PH")
    if key not in risk_store:
        raise KeyError(f"Missing saved Cox risk record: {key}")
    return risk_store[key]

for feature_set in RESULT2_FEATURES:
    record = result2_record(feature_set)
    if len(record["train_risk"]) != len(df_train) or len(record["test_risk"]) != len(df_test):
        raise RuntimeError(f"Risk/data length mismatch for {feature_set}")

result2_horizon_audit = horizon_support_audit(df_train, df_test, HORIZONS)
result2_horizon_audit["Evaluation_Population"] = "locked test set"
result2_horizon_audit.to_csv(
    os.path.join(RESULT2_SOURCE_DIR, "horizon_support_audit.csv"),
    index=False,
)
display(result2_horizon_audit)

competing_event_column = os.environ.get("CHD_COMPETING_EVENT_COL", "").strip()
competing_audit = {
    "Competing_Event_Column_Configured": competing_event_column,
    "Competing_Event_Column_Available": bool(
        competing_event_column and competing_event_column in df_test.columns
    ),
    "Current_Outcome_Columns": "Time;Event",
    "Primary_Estimand": "Cause-specific time to first CHD under right censoring",
    "Warning": (
        "A competing-risk analysis requires a separate non-CHD death indicator. "
        "Event=0 alone cannot distinguish death from loss to follow-up."
    ),
}
pd.DataFrame([competing_audit]).to_csv(
    os.path.join(RESULT2_SOURCE_DIR, "competing_risk_data_audit.csv"),
    index=False,
)
if competing_audit["Competing_Event_Column_Available"]:
    warnings.warn(
        "A competing-event column was detected. This notebook remains the "
        "right-censored primary analysis; add a competing-risk sensitivity "
        "analysis rather than silently recoding the endpoint."
    )


In [ ]:
# Fit only the train-based absolute-risk calibration layer needed for NRI/IDI,
# IBS and DCA. The Cox models were fitted or resumed above; this cell fits only the training calibration layer.
result2_calibrators = {}
result2_predictions = {}
result2_ibs_objects = {}

for feature_set in RESULT2_FEATURES:
    record = result2_record(feature_set)
    calibrator = fit_absolute_risk_calibrator(
        df_train,
        record["train_risk"],
        label=f"{feature_set} | Cox PH",
    )
    result2_calibrators[feature_set] = calibrator
    for horizon in HORIZONS:
        result2_predictions[(feature_set, horizon)] = predict_absolute_risk(
            calibrator,
            record["test_risk"],
            horizon,
        )
        result2_ibs_objects[(feature_set, horizon)] = subject_level_ipcw_ibs(
            df_train,
            df_test,
            calibrator,
            record["test_risk"],
            horizon,
        )

prediction_manifest = pd.DataFrame([
    {
        "Feature_Set": feature_set,
        "Model": "Cox PH",
        "Split": "test",
        "Horizon_Years": horizon,
        "Prediction_N": len(result2_predictions[(feature_set, horizon)]),
        "Prediction_Mean": float(np.mean(result2_predictions[(feature_set, horizon)])),
        "Calibration_Method": result2_calibrators[feature_set]["method"],
        "Train_Eid_SHA256": eid_digest(df_train),
        "Test_Eid_SHA256": eid_digest(df_test),
    }
    for feature_set in RESULT2_FEATURES
    for horizon in HORIZONS
])
prediction_manifest.to_csv(
    os.path.join(RESULT2_SOURCE_DIR, "absolute_risk_prediction_manifest.csv"),
    index=False,
)


In [ ]:
# Parallel Result 2 source-data generation. Statistical estimands are unchanged.
from joblib import Parallel, delayed
import time

RESULT2_PRIMARY_COMPARISONS = [
    ("Clinical + PRS vs Clinical", "Clinical_plus_PRS", "Clinical_only", "Extension vs Clinical"),
    ("Clinical + NMR_score vs Clinical", "Clinical_plus_NMR_score", "Clinical_only", "Extension vs Clinical"),
    ("Clinical + PRS + NMR_score vs Clinical", "Clinical_plus_PRS_plus_NMR_score", "Clinical_only", "Extension vs Clinical"),
]
RESULT2_DIRECT_COMPARISONS = [
    ("Integrated vs Clinical + PRS", "Clinical_plus_PRS_plus_NMR_score", "Clinical_plus_PRS", "Integrated vs single extension"),
    ("Integrated vs Clinical + NMR_score", "Clinical_plus_PRS_plus_NMR_score", "Clinical_plus_NMR_score", "Integrated vs single extension"),
]
RESULT2_ALL_COMPARISONS = RESULT2_PRIMARY_COMPARISONS + RESULT2_DIRECT_COMPARISONS
RESULT2_MAX_JOBS = int(os.environ.get("CHD_RESULT2_N_JOBS", str(min(10, CPU_WORKERS))))
if RESULT2_MAX_JOBS < 1:
    raise ValueError("CHD_RESULT2_N_JOBS must be at least 1")

def _parallel(tasks, label):
    if not tasks:
        return []
    workers = min(RESULT2_MAX_JOBS, len(tasks))
    print(f"{label}: {len(tasks)} tasks | {workers} workers", flush=True)
    started = time.time()
    values = Parallel(n_jobs=workers, backend="loky", verbose=10, max_nbytes="50M")(tasks)
    print(f"{label} complete in {(time.time() - started) / 60:.1f} min", flush=True)
    return values

def _uno_job(label, new_feature, reference_feature, group, horizon, seed):
    row = paired_bootstrap_uno(
        df_train, df_test, result2_record(new_feature)["test_risk"],
        result2_record(reference_feature)["test_risk"], horizon, seed=seed,
    )
    return {"Comparison": label, "Comparison_Group": group, "Feature_Set": new_feature,
            "Reference_Feature_Set": reference_feature, "Model": "Cox PH", "Split": "test",
            "Horizon_Years": horizon, **row}

def _harrell_job(label, new_feature, reference_feature, group, seed):
    row = paired_bootstrap_harrell(
        df_test, result2_record(new_feature)["test_risk"],
        result2_record(reference_feature)["test_risk"], seed=seed,
    )
    return {"Comparison": label, "Comparison_Group": group, "Feature_Set": new_feature,
            "Reference_Feature_Set": reference_feature, "Model": "Cox PH", "Split": "test", **row}

def _ibs_job(label, new_feature, reference_feature, group, horizon, seed):
    row = paired_bootstrap_delta_ibs(
        result2_ibs_objects[(new_feature, horizon)],
        result2_ibs_objects[(reference_feature, horizon)], seed=seed,
    )
    return {"Comparison": label, "Comparison_Group": group, "Feature_Set": new_feature,
            "Reference_Feature_Set": reference_feature, "Model": "Cox PH", "Split": "test",
            "Horizon_Years": horizon,
            "Delta_Definition": "IBS_new_minus_IBS_reference; lower_is_better", **row}

def _nri_job(label, new_feature, reference_feature, group, horizon, seed):
    row = bootstrap_ipcw_nri_idi(
        df_test, result2_predictions[(reference_feature, horizon)],
        result2_predictions[(new_feature, horizon)], horizon,
        cutpoints=NRI_CUTPOINTS, seed=seed,
    )
    return {"Comparison": label, "Comparison_Group": group, "Feature_Set": new_feature,
            "Reference_Feature_Set": reference_feature, "Model": "Cox PH", "Split": "test",
            "Horizon_Years": horizon,
            "NRI_Cutpoints": ";".join(str(value) for value in NRI_CUTPOINTS),
            "NRI_Cutpoint_Source": NRI_CUTPOINT_SOURCE,
            "Censoring_Method": "IPCW; censoring KM refitted within each test bootstrap", **row}

result2_a = pd.DataFrame(_parallel([
    delayed(_uno_job)(label, new, ref, group, horizon, BOOTSTRAP_SEED + 1000 + i * 10 + j)
    for i, (label, new, ref, group) in enumerate(RESULT2_ALL_COMPARISONS)
    for j, horizon in enumerate(HORIZONS)
], "Result 2 Uno IPCW bootstrap"))
result2_harrell = pd.DataFrame(_parallel([
    delayed(_harrell_job)(label, new, ref, group, BOOTSTRAP_SEED + 1100 + i)
    for i, (label, new, ref, group) in enumerate(RESULT2_ALL_COMPARISONS)
], "Result 2 Harrell bootstrap"))
result2_c = pd.DataFrame(_parallel([
    delayed(_ibs_job)(label, new, ref, group, horizon, BOOTSTRAP_SEED + 1200 + i * 10 + j)
    for i, (label, new, ref, group) in enumerate(RESULT2_ALL_COMPARISONS)
    for j, horizon in enumerate(HORIZONS)
], "Result 2 IPCW IBS bootstrap"))
result2_d = pd.DataFrame(_parallel([
    delayed(_nri_job)(label, new, ref, group, horizon, BOOTSTRAP_SEED + 1300 + i * 10 + j)
    for i, (label, new, ref, group) in enumerate(RESULT2_ALL_COMPARISONS)
    for j, horizon in enumerate(HORIZONS)
], "Result 2 IPCW NRI/IDI bootstrap"))

result2_a.to_csv(os.path.join(RESULT2_SOURCE_DIR, "figure2A_delta_uno_cindex.csv"), index=False)
result2_harrell.to_csv(os.path.join(RESULT2_SOURCE_DIR, "supplement_delta_harrell_cindex.csv"), index=False)
result2_c.to_csv(os.path.join(RESULT2_SOURCE_DIR, "figure2C_delta_ipcw_ibs.csv"), index=False)
result2_d.to_csv(os.path.join(RESULT2_SOURCE_DIR, "figure2D_ipcw_nri_idi.csv"), index=False)

result2_dca_labels = {
    "Clinical": "Clinical_only", "Clinical + PRS": "Clinical_plus_PRS",
    "Clinical + NMR_score": "Clinical_plus_NMR_score",
    "Clinical + PRS + NMR_score": "Clinical_plus_PRS_plus_NMR_score",
}
result2_e = pd.concat([
    survival_dca(df_test, {
        label: result2_predictions[(feature_set, horizon)]
        for label, feature_set in result2_dca_labels.items()
    }, horizon)
    for horizon in HORIZONS
], ignore_index=True)
result2_e.to_csv(os.path.join(RESULT2_SOURCE_DIR, "figure2E_survival_dca.csv"), index=False)
print("Result 2 source data complete.", flush=True)
display(result2_a)
display(result2_c)
display(result2_d)


In [ ]:
# Figure 2A: Uno IPCW Delta C-index. One figure, no multi-panel layout.
comparison_order = [item[0] for item in RESULT2_PRIMARY_COMPARISONS + RESULT2_DIRECT_COMPARISONS]
fig, ax = plt.subplots(figsize=(9.0, 5.3))
y = np.arange(len(comparison_order))
horizon_style = {
    5.0: (CNS["teal"], "o", -0.11, "5 years"),
    10.0: (CNS["blue"], "s", 0.11, "10 years"),
}
for horizon, (color, marker, offset, legend_label) in horizon_style.items():
    data = result2_a.loc[np.isclose(result2_a["Horizon_Years"], horizon)].copy()
    data["Order"] = data["Comparison"].map({label: index for index, label in enumerate(comparison_order)})
    data = data.sort_values("Order")
    estimate = data["Delta_Uno_C"].to_numpy(dtype=float)
    lower = data["Delta_Uno_C_Lower95"].to_numpy(dtype=float)
    upper = data["Delta_Uno_C_Upper95"].to_numpy(dtype=float)
    ax.errorbar(
        estimate,
        y + offset,
        xerr=[np.maximum(0, estimate - lower), np.maximum(0, upper - estimate)],
        fmt=marker,
        color=color,
        ecolor=color,
        capsize=3,
        markersize=6,
        label=legend_label,
    )
ax.axvline(0, color=CNS["gray"], lw=1)
ax.axhline(len(RESULT2_PRIMARY_COMPARISONS) - 0.5, color=CNS["light_gray"], lw=1)
ax.set_yticks(y)
ax.set_yticklabels(comparison_order)
ax.invert_yaxis()
ax.set_xlabel("Paired delta Uno IPCW C-index (95% CI)")
ax.set_title("Figure 2A. Incremental discrimination", loc="left", fontweight="bold")
ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()
save_figure(fig, RESULT2_FIGURE_DIR, "figure2A_delta_uno_ipcw_cindex")
plt.show()


In [ ]:
# Figure 2B: both HRs must come from the same saved joint Cox fit.
hr_source_path = os.path.join(
    SENSITIVITY_OUTPUT_DIR,
    "04_results",
    "tables",
    "result2_cox_prs_nmrscore_hr.csv",
)
if not os.path.exists(hr_source_path):
    raise FileNotFoundError(
        "Saved joint-model HR table not found: " + hr_source_path + ". "
        "The joint HR table must have been generated by the fit cell above."
    )
result2_b = pd.read_csv(hr_source_path)
required_hr_columns = {"Feature_Set", "Variable", "HR", "HR_CI_Lower", "HR_CI_Upper"}
if not required_hr_columns.issubset(result2_b.columns):
    raise RuntimeError("Unexpected HR table columns: " + ", ".join(result2_b.columns))
result2_b = result2_b.loc[
    result2_b["Feature_Set"].astype(str).eq("Clinical_plus_PRS_plus_NMR_score")
].copy()
if len(result2_b) != 2 or result2_b["Feature_Set"].nunique() != 1:
    raise RuntimeError("PRS and NMR_score HRs are not two rows from one joint Cox model.")
result2_b["Display_Label"] = result2_b["Variable"].replace({
    PRS_COL: "CAD PRS (per SD)",
    "NMR_score": "NMR_score (per SD)",
})
result2_b["Joint_Model"] = "Clinical + PRS + NMR_score"
result2_b["CI_Excludes_1"] = (
    (result2_b["HR_CI_Lower"].astype(float) > 1)
    | (result2_b["HR_CI_Upper"].astype(float) < 1)
)
result2_b.to_csv(
    os.path.join(RESULT2_SOURCE_DIR, "figure2B_joint_model_adjusted_hr.csv"),
    index=False,
)

plot_b = result2_b.sort_values("Display_Label").reset_index(drop=True)
CNS["red"] = "#C44E52"
fig, ax = plt.subplots(figsize=(7.4, 3.6))
y = np.arange(len(plot_b))
estimate = plot_b["HR"].to_numpy(dtype=float)
lower = plot_b["HR_CI_Lower"].to_numpy(dtype=float)
upper = plot_b["HR_CI_Upper"].to_numpy(dtype=float)
ax.errorbar(
    estimate,
    y,
    xerr=[np.maximum(0, estimate - lower), np.maximum(0, upper - estimate)],
    fmt="o",
    color=CNS["red"],
    ecolor=CNS["red"],
    capsize=3,
    markersize=7,
)
ax.axvline(1, color=CNS["gray"], lw=1)
ax.set_xscale("log")
ax.set_yticks(y)
ax.set_yticklabels(plot_b["Display_Label"])
ax.invert_yaxis()
ax.set_xlabel("Adjusted hazard ratio per 1-SD increment (95% CI)")
ax.set_title("Figure 2B. Adjusted associations in the joint Cox model", loc="left", fontweight="bold")
ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
clean_axes(ax)
fig.tight_layout()
save_figure(fig, RESULT2_FIGURE_DIR, "figure2B_joint_model_adjusted_hr")
plt.show()


In [ ]:
# Figure 2C: paired IPCW Delta IBS at 5 and 10 years.
comparison_order_c = [item[0].replace(" vs Clinical", "") for item in RESULT2_ALL_COMPARISONS]
plot_c = result2_c.copy()
plot_c["Display_Label"] = plot_c["Comparison"].str.replace(" vs Clinical", "", regex=False)
scale = 1e5 if np.nanmax(np.abs(plot_c[["Delta_IBS", "Delta_IBS_Lower95", "Delta_IBS_Upper95"]])) < 1e-3 else 1.0
scale_label = " (x10^-5)" if scale == 1e5 else ""
fig, ax = plt.subplots(figsize=(10.5, 6.6))
y = np.arange(len(comparison_order_c))
for horizon, (color, marker, offset, legend_label) in horizon_style.items():
    data = plot_c.loc[np.isclose(plot_c["Horizon_Years"], horizon)].copy()
    data["Order"] = data["Display_Label"].map({label: index for index, label in enumerate(comparison_order_c)})
    data = data.sort_values("Order")
    estimate = data["Delta_IBS"].to_numpy(dtype=float) * scale
    lower = data["Delta_IBS_Lower95"].to_numpy(dtype=float) * scale
    upper = data["Delta_IBS_Upper95"].to_numpy(dtype=float) * scale
    ax.errorbar(
        estimate,
        y + offset,
        xerr=[np.maximum(0, estimate - lower), np.maximum(0, upper - estimate)],
        fmt=marker,
        color=color,
        ecolor=color,
        capsize=3,
        markersize=6,
        label=legend_label,
    )
ax.axvline(0, color=CNS["gray"], lw=1)
ax.set_yticks(y)
ax.set_yticklabels(comparison_order_c)
ax.invert_yaxis()
ax.set_xlabel(f"Delta IPCW IBS: new minus reference{scale_label} (lower is better)")
ax.set_title("Figure 2C. Prediction error", loc="left", fontweight="bold")
ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
ax.legend(frameon=False)
clean_axes(ax)
fig.tight_layout()
save_figure(fig, RESULT2_FIGURE_DIR, "figure2C_delta_ipcw_ibs")
plt.show()


In [ ]:
# Figure 2D: one reclassification figure per horizon.
for horizon in HORIZONS:
    data = result2_d.loc[np.isclose(result2_d["Horizon_Years"], horizon)].copy()
    data["Display_Label"] = data["Comparison"].str.replace(" vs Clinical", "", regex=False)
    data = data.set_index("Display_Label").reindex(comparison_order_c).reset_index()
    fig, ax = plt.subplots(figsize=(10.5, 6.8))
    y = np.arange(len(data))
    for metric, low_col, high_col, color, marker, offset, label in [
        ("NRI_Categorical", "NRI_Categorical_Lower95", "NRI_Categorical_Upper95", CNS["green"], "o", -0.12, "Categorical NRI"),
        ("IDI", "IDI_Lower95", "IDI_Upper95", CNS["orange"], "s", 0.12, "IDI"),
    ]:
        estimate = data[metric].to_numpy(dtype=float)
        lower = data[low_col].to_numpy(dtype=float)
        upper = data[high_col].to_numpy(dtype=float)
        ax.errorbar(
            estimate,
            y + offset,
            xerr=[np.maximum(0, estimate - lower), np.maximum(0, upper - estimate)],
            fmt=marker,
            color=color,
            ecolor=color,
            capsize=3,
            markersize=6,
            label=label,
        )
    ax.axvline(0, color=CNS["gray"], lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels(data["Display_Label"])
    ax.invert_yaxis()
    ax.set_xlabel("IPCW estimate: new model vs stated reference (95% CI)")
    ax.set_title(f"Figure 2D. {horizon:g}-year reclassification", loc="left", fontweight="bold")
    ax.grid(axis="x", color=CNS["light_gray"], lw=0.8)
    ax.legend(frameon=False)
    clean_axes(ax)
    fig.tight_layout()
    save_figure(fig, RESULT2_FIGURE_DIR, f"figure2D_{int(horizon)}y_ipcw_nri_idi")
    plt.show()


In [ ]:
# Supplementary calibration: one separate figure per horizon.
result2_calibration_parts = []
for feature_set in RESULT2_FEATURES:
    for horizon in HORIZONS:
        grouped = calibration_by_quantile(
            df_test,
            result2_predictions[(feature_set, horizon)],
            horizon,
            groups=10,
        )
        grouped.insert(0, "Feature_Set", feature_set)
        grouped.insert(1, "Model", "Cox PH")
        grouped.insert(2, "Split", "test")
        grouped.insert(3, "Horizon_Years", float(horizon))
        result2_calibration_parts.append(grouped)

result2_calibration = pd.concat(result2_calibration_parts, ignore_index=True)
atomic_write_csv(
    result2_calibration,
    os.path.join(RESULT2_SOURCE_DIR, "supplement_result2_calibration_by_decile.csv"),
    index=False,
)

result2_feature_labels = {
    "Clinical_only": "Clinical",
    "Clinical_plus_PRS": "Clinical + PRS",
    "Clinical_plus_NMR_score": "Clinical + NMR_score",
    "Clinical_plus_PRS_plus_NMR_score": "Clinical + PRS + NMR_score",
}
result2_colors = {
    "Clinical_only": CNS["black"],
    "Clinical_plus_PRS": CNS["blue"],
    "Clinical_plus_NMR_score": CNS["green"],
    "Clinical_plus_PRS_plus_NMR_score": CNS["red"],
}
for horizon in HORIZONS:
    fig, ax = plt.subplots(figsize=(8.4, 6.4))
    for feature_set in RESULT2_FEATURES:
        data = result2_calibration.loc[
            result2_calibration["Feature_Set"].eq(feature_set)
            & np.isclose(result2_calibration["Horizon_Years"], horizon)
        ].sort_values("Predicted_Risk")
        ax.plot(
            data["Predicted_Risk"], data["Observed_KM_Risk"],
            marker="o", markersize=5.5, linewidth=2.0,
            color=result2_colors[feature_set], label=result2_feature_labels[feature_set],
        )
    limit = float(max(0.01, result2_calibration.loc[
        np.isclose(result2_calibration["Horizon_Years"], horizon),
        ["Predicted_Risk", "Observed_KM_Risk"],
    ].to_numpy(float).max() * 1.05))
    ax.plot([0, limit], [0, limit], linestyle="--", color=CNS["gray"], linewidth=1.2, label="Ideal")
    ax.set_xlim(0, limit); ax.set_ylim(0, limit)
    ax.set_xlabel("Mean predicted CHD risk")
    ax.set_ylabel("Kaplan-Meier observed CHD risk")
    ax.set_title(f"Supplementary Figure. {horizon:g}-year calibration", loc="left", fontweight="bold")
    ax.grid(color=CNS["light_gray"], linewidth=0.8)
    ax.legend(frameon=False, fontsize=11.0)
    clean_axes(ax); fig.tight_layout()
    save_figure(fig, RESULT2_FIGURE_DIR, f"supplement_result2_{int(horizon)}y_calibration")
    plt.show()


In [ ]:
# Figure 2E: one standard survival-DCA figure per horizon.
dca_colors = {
    "Clinical": CNS["black"],
    "Clinical + PRS": CNS["blue"],
    "Clinical + NMR_score": CNS["green"],
    "Clinical + PRS + NMR_score": CNS["red"],
    "Treat all": CNS["gray"],
    "Treat none": CNS["gray"],
}
for horizon in HORIZONS:
    data = result2_e.loc[np.isclose(result2_e["Horizon_Years"], horizon)].copy()
    draw_dca(
        data,
        title=f"Figure 2E. {horizon:g}-year survival decision curve analysis",
        output_dir=RESULT2_FIGURE_DIR,
        stem=f"figure2E_{int(horizon)}y_survival_dca",
        colors=dca_colors,
    )


In [ ]:
# Supplementary Harrell C sensitivity figure.
draw_forest(
    result2_harrell,
    estimate="Delta_Harrell_C",
    lower="Delta_Harrell_C_Lower95",
    upper="Delta_Harrell_C_Upper95",
    label="Comparison",
    title="Supplementary Figure. Harrell C-index sensitivity analysis",
    xlabel="Paired delta Harrell C-index (95% CI)",
    output_dir=RESULT2_FIGURE_DIR,
    stem="supplement_delta_harrell_cindex",
    color=CNS["purple"],
)

print("Result 2 complete. No primary Cox model was retrained.")
print("Figures:", RESULT2_FIGURE_DIR)
print("Source data:", RESULT2_SOURCE_DIR)


In [ ]:
def _pipeline_context(manifest):
    nested = manifest.get("analysis_context", {}) if isinstance(manifest, dict) else {}
    cohort_id = str(manifest.get("cohort_id", nested.get("cohort_id", "unspecified")))
    analysis_role = str(manifest.get("analysis_role", nested.get("analysis_role", "unspecified")))
    if cohort_id.lower() == "coverage90" and analysis_role.lower() == "primary":
        raise RuntimeError("coverage90 must remain a sensitivity cohort; pipeline manifest labels it primary")
    return cohort_id, analysis_role

cohort_id, analysis_role = _pipeline_context(pipeline_manifest)
context_table = pd.DataFrame([{
    "Result": "Result2", "Cohort_ID": cohort_id, "Analysis_Role": analysis_role,
    "Primary_Model": "unpenalized Cox PH", "Model_Count": 4,
}])
atomic_write_csv(context_table, os.path.join(RESULT2_SOURCE_DIR, "result2_analysis_context.csv"), index=False)
atomic_write_json({
    "status": "success", "completed_at": str(pd.Timestamp.now()),
    "pipeline_manifest_sha256": current_pipeline_sha256(),
    "cohort_id": cohort_id, "analysis_role": analysis_role,
    "primary_model": "unpenalized Cox PH", "model_count": 4,
    "test_used_for_tuning": False,
    "required_outputs": ["Uno IPCW C", "Harrell C", "IPCW IBS", "NRI", "IDI", "calibration", "survival DCA"],
}, os.path.join(RESULT2_ROOT, "SUCCESS.json"))
